[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/particle_charges/particle_charges.ipynb)

# Spin(10) Particle Charges

This example computes the Standard Model charges and gauge connections across one generation of quarks and leptons inside a chiral Spin(10) spinor. One generation of matter is modelled as a minimal left ideal of ten internal directions, where colour and weak rotations connect members within each multiplet while mixed rotations connect quarks with leptons.

Left multiplication keeps the minimal left ideal invariant so `action = projection(Even * State)` turns every internal bivector and commuting projector into a state map, the ten-direction `pseudoscalar` acts as a common quarter turn pairing real coefficients into physical amplitudes, and `-pseudoscalar * planes` splits the spinor into definite-charge sectors whose gauge couplings follow from `blocks.scalar_norm_squared()`.

In the exterior-algebra grand-unified notation of [Baez and Huerta](https://arxiv.org/html/0904.1556#S3.SS5), the chiral $\mathbf{16}$ spinor of $\operatorname{Spin}(10)$ is built over $\mathbb{C}$ as $\Lambda^{\text{even}}(\mathbb{C}^5)$ using five complexified fermionic creation and annihilation operators $f_k^\dagger = \tfrac{1}{2}(\Gamma_{2k-1} + i\,\Gamma_{2k})$ and $f_k = \tfrac{1}{2}(\Gamma_{2k-1} - i\,\Gamma_{2k})$ acting on a Fock vacuum $|0\rangle$, with occupation operators $N_k = f_k^\dagger f_k$, electric charge $Q = T_3 + Y$, and $\mathfrak{su}(3)_c \oplus \mathfrak{su}(2)_L \oplus \mathfrak{u}(1)_Y$ transition elements evaluated as complex $16 \times 16$ matrix elements $\sum_a |\langle s' | T^a | s \rangle|^2$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from functools import reduce
from operator import mul

import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.util.animation import save_figure
from examples.quantum.particle_charges import render

ga = Algebra("a+b+c+d+e+f+g+h+i+j+")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                      # Scalar+
Bivector = ga.gatype.bivector()                                  # Bivector+
Even = ga.gatype.even()                                          # Even+
Full = ga.gatype.full()                                          # Full+
# One independent blade from each group of blades the ideal identifies.
State = Full.derive.within(Algebra("b+c+d+f+j+"))                # Full+
StateMap = ga.gatype((State, State))                             # State+ <- State+

## Minimal left ideal and spinor chart

Internal rotations keep a minimal left ideal invariant under left multiplication. Five commuting square-one projectors narrow the algebra to an ideal of 32 real coefficients, while the ten-direction `pseudoscalar` squares to minus one and commutes with every even element, turning each pair of coefficients into one physical amplitude.

In [ ]:
# Five commuting square-one factors select a minimal left ideal with 32 real coefficients.
ideal = (1 + mv.a) * (1 + mv.bcde) * (1 + mv.bcfg) * (1 + mv.bdfh) * (1 + mv.cdfi) / 32  # [] Full+
# Each chart blade appears with weight 1/32 inside the ideal.
readout = (32 * Full).cast(State)                                # [] State+ <- Full+
# Right multiplication by the ideal carries any element into it; the chart reads its coefficients.
projection = readout(Full * ideal)                               # [] State+ <- Full+
# Left multiplication keeps the ideal invariant, turning each even element into a state map.
action = projection(Even * State)                                # [] State+ <- (Even+, State+)
pseudoscalar = mv.abcdefghij                                     # [] Even+

## Charge sectors

Five orthogonal bivector planes act on the spinor as commuting quarter turns alongside `pseudoscalar`. Multiplying each plane by `-pseudoscalar` produces a commuting sign operator whose eigenvalues distinguish states turning with `pseudoscalar` from states turning against it. The first three planes carry colour and the last two carry weak isospin, so fixing four signs isolates one particle sector while the fifth sign follows from chirality.

In [ ]:
from examples.quantum.particle_charges.scenarios import hypercharge_weights, occupations

In [ ]:
def sector_occupations(planes: Bivector) -> StateMap:
    """Each commuting plane compares its spinor quarter turn against the pseudoscalar to count occupation (`[...] State+ <- State+`)."""
    signs = -pseudoscalar * planes                               # [...] Even+
    return (State - action(signs)) / 2                           # [...] State+ <- State+


def sector_projectors(planes: Bivector, occupations: np.ndarray) -> StateMap:
    """Four commuting sign factors isolate each two-dimensional particle sector (`[...] State+ <- State+`)."""
    signs = -pseudoscalar * planes                               # [5] Even+
    factors = (1 + signs[None, :4] * (1 - 2 * occupations[:, :4])) / 2  # [..., 4] Even+
    selectors = reduce(mul, (factors[:, index] for index in range(4)))  # [...] Even+
    return action(selectors)                                     # [...] State+ <- State+


planes = stack((mv.ab, mv.cd, mv.ef, mv.gh, mv.ij))              # [5] Bivector+
occupation = sector_occupations(planes)                          # [5] State+ <- State+
projectors = sector_projectors(planes, occupations)              # [16] State+ <- State+

In [ ]:
def sector_value(observable: StateMap, projectors: StateMap) -> Scalar:
    """A commuting observable takes a constant trace ratio on each particle sector (`[...] Scalar+`)."""
    return observable(projectors).trace() / projectors.trace()   # [...] Scalar+


hypercharge = (hypercharge_weights * occupation).sum(axis=0)     # [] State+ <- State+
left_isospin = (occupation[3] - occupation[4]) / 2               # [] State+ <- State+
right_isospin = (occupation[3] + occupation[4] - State) / 2      # [] State+ <- State+
electric_charge = hypercharge + left_isospin                     # [] State+ <- State+
baryon_minus_lepton = State - 2 * occupation[:3].sum(axis=0) / 3 # [] State+ <- State+
# Two commuting colour observables separate the three quark colours within each multiplet.
colour_observables = stack((
    (occupation[0] - occupation[1]) / 2,
    (occupation[0] + occupation[1] - 2 * occupation[2]) / (2 * np.sqrt(3)),
))                                                               # [2] State+ <- State+

charges = sector_value(electric_charge, projectors)              # [16] Scalar+
hypercharges = sector_value(hypercharge, projectors)             # [16] Scalar+
left_isospins = sector_value(left_isospin, projectors)           # [16] Scalar+
right_isospins = sector_value(right_isospin, projectors)         # [16] Scalar+
baryon_minus_leptons = sector_value(baryon_minus_lepton, projectors)  # [16] Scalar+
colour_weights = sector_value(colour_observables[None, :], projectors[:, None])  # [16, 2] Scalar+

In [ ]:
print("charges:", charges.to_array())
print("hypercharges:", hypercharges.to_array())
print("left isospins:", left_isospins.to_array())

The sixteen sectors group into eight particle multiplets, including charge-conjugate right-handed fields with opposite charges and a neutral right-handed neutrino singlet.

In Lie-algebra weight notation, hypercharge decomposes across the left-right subgroup $\mathfrak{su}(4)_C \oplus \mathfrak{su}(2)_L \oplus \mathfrak{su}(2)_R$ as $Y = T_{3R} + \tfrac{1}{2}(B - L)$, so $Q = T_{3L} + T_{3R} + \tfrac{1}{2}(B - L)$.

## Colour and weak actions

Paired bivector turns exchange occupations between two planes while preserving their sum. Colour generators mix the three colours within each quark multiplet while leaving leptons untouched, and weak generators mix the two members of each left-handed doublet while leaving conjugate singlets untouched.

In [ ]:
def transition_strength(generators: StateMap, projectors: StateMap) -> Scalar:
    """The squared gauge blocks sum the transition strength between every pair of particle sectors (`[..., ...] Scalar+`)."""
    blocks = projectors[:, None, None](generators[None, None, :](projectors[None, :, None]))  # [..., ..., ...] State+ <- State+
    # Two real directions paired by the pseudoscalar carry one physical amplitude.
    return blocks.scalar_norm_squared().contract().sum(axis=-1) / 2  # [..., ...] Scalar+


# Paired turns exchange two occupations while preserving their sum.
colour_bivectors = stack((
    mv.ac + mv.bd, mv.ad - mv.bc,
    mv.ae + mv.bf, mv.af - mv.be,
    mv.ce + mv.df, mv.cf - mv.de,
    mv.ab - mv.cd,
    (mv.ab + mv.cd - 2 * mv.ef) / np.sqrt(3),
)) / 4                                                           # [8] Bivector+
weak_bivectors = stack((mv.gi + mv.hj, mv.gj - mv.hi, mv.gh - mv.ij)) / 4  # [3] Bivector+
colour_generators = action(colour_bivectors)                     # [8] State+ <- State+
weak_generators = action(weak_bivectors)                         # [3] State+ <- State+

colour_strength = transition_strength(colour_generators, projectors)  # [16, 16] Scalar+
weak_strength = transition_strength(weak_generators, projectors)  # [16, 16] Scalar+
gauge_figure = render.connections_figure(
    occupations, left_isospins, right_isospins, colour_weights,
    stack((colour_strength, weak_strength)),
)
save_figure(gauge_figure, "particle_charges_gauge")
gauge_figure

The left panel shows colour connections and the right panel shows weak connections across the sixteen sectors, placed by their weak isospins and tilted colour weights. Red, green, and blue markers denote quarks while gold markers denote leptons; filled markers indicate left-handed doublets in the horizontal clusters, hollow markers indicate conjugate singlets in the vertical clusters, and strokes connect sector pairs with nonzero generator action.

## Connections between multiplets

Additional Spin(10) bivectors pair a colour direction with a weak direction, exchanging a colour occupation for a weak occupation and linking quark sectors directly to lepton sectors within the same spinor.

In [ ]:
mixing_bivector = (mv.ag + mv.bh) / 4                            # [] Bivector+
mixing_generators = action(mixing_bivector)[None]                # [1] State+ <- State+
mixing_strength = transition_strength(mixing_generators, projectors)  # [16, 16] Scalar+
unified_figure = render.connections_figure(
    occupations, left_isospins, right_isospins, colour_weights, mixing_strength[None],
)
save_figure(unified_figure, "particle_charges_unified")
unified_figure

Orange strokes cross the multiplet boundaries preserved by colour and weak rotations, linking gold lepton markers to coloured quark markers and connecting quark sectors of opposite weak assignment.

In [ ]:
# checks
assert ((charges - hypercharges - left_isospins).abs() < 1e-10).all()
assert ((hypercharges - right_isospins - baryon_minus_leptons / 2).abs() < 1e-10).all()
assert ((projectors.trace() - 2).abs() < 1e-10).all()
edge_counts = [
    int(np.triu(strength > 1e-10, k=1).sum())
    for strength in (colour_strength, weak_strength, mixing_strength)
]
assert edge_counts == [12, 4, 4]